In [9]:
import os
import time
from libtmux import Server

exp_name = "Qwen3-4B-kvINT4"
base_port = 30501

# tmux -S /data/jisenli2/.tmux-secure-4B-30501 attach -t r2egym
# tmux -S /data/jisenli2/.tmux-secure-4B-30501 attach -t docker

In [2]:
original_model_path = "/data/shared/charlie/GLM4.7-sft-mix"
huggingface_path = "huggingface_10000/"
model_assets_path = "model_assets/"
nodes = [
    "research-secure-04",
    "research-secure-06",
]

# If a teammate already launched sglang elsewhere, point this at that
# OpenAI-compatible endpoint and keep local tmux launch disabled.
# sglang_endpoint = "http://research-common-33:30100/v1"
sglang_api_key = "sk-dummy-key-for-local-model"
launch_sglang_in_tmux = False

# openai/Qwen/Qwen3-Coder-sft-mix-14000
# rebench-verify

In [3]:
import os
from libtmux import Server

tmux_socket_path = f"/data/jisenli2/.tmux-secure-4B-{base_port}"
server = Server(socket_path=tmux_socket_path)

sessions = server.sessions
print(f"Found {len(sessions)} sessions to delete")

for session in sessions:
    try:
        session_name = session.name
        session.kill()
        print(f"✅ Session '{session_name}' deleted successfully")
    except Exception as e:
        print(f"❌ Error deleting session: {e}")

print("All sessions deleted!")

Found 0 sessions to delete
All sessions deleted!


# sglang

In [4]:
# connect to the docker server
session_name = "sglang"
# tmux_socket_path = "/data/jisenli2/.tmux-secure-multi"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [5]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Attempt 1: SSHing to research-secure-04
Successfully connected to research-secure-04
Waiting for pane to be ready: 06
Successfully connected to research-secure-06


# docker

In [6]:
# connect to the docker server
session_name = "docker"
# tmux_socket_path = "/data/jisenli2/.tmux-secure-multi"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [7]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Attempt 1: SSHing to research-secure-04
Successfully connected to research-secure-04
Waiting for pane to be ready: 06
Successfully connected to research-secure-06


In [8]:
CMD = """
sudo usermod -aG docker $USER
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 04
  ✅ Command sent to 06


In [9]:
CMD = """
newgrp docker
"""

# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")


  ✅ Command sent to 04
  ✅ Command sent to 06


In [10]:
CMD = """
source /data/jisenli2/miniconda/etc/profile.d/conda.sh
conda activate r2egym
cd /data/jisenli2/R2E-Gym/scripts
python docker_storage_monitor.py
"""
# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 04
  ✅ Command sent to 06


# r2egym

In [11]:
# connect to the docker server
session_name = "r2egym"
# tmux_socket_path = "/data/jisenli2/.tmux-secure-multi"

# Create the tmux socket directory if it doesn't exist
tmux_socket_dir = os.path.dirname(tmux_socket_path)
print(f"Creating tmux socket directory: {tmux_socket_dir}")
os.makedirs(tmux_socket_dir, exist_ok=True)


server = Server(socket_path=tmux_socket_path)
session = server.sessions.get(session_name=session_name, default=None)
if not session:
    session = server.new_session(session_name)

Creating tmux socket directory: /data/jisenli2


In [12]:
window_names = [f"{node.split('-')[-1]}" for node in nodes]


# Create windows for each node and SSH into them
for i, node in enumerate(nodes):
    window_name = f"{node.split('-')[-1]}"
    
    # Check if window already exists
    existing_window = None
    for w in session.windows:
        if w.name == window_name:
            existing_window = w
            break
    
    if existing_window:
        # Use existing window
        window = existing_window
        print(f"Using existing window: {window_name}")
    elif i == 0:
        # Use the existing first window for the first node
        window = session.windows[0]
        window.rename_window(window_name)
    else:
        # Create new windows for subsequent nodes
        window = session.new_window(window_name=window_name, attach=False)

    # Get the pane and SSH into the node
    pane = window.active_pane

    while not pane.capture_pane():
        print(f"Waiting for pane to be ready: {window_name}")
        time.sleep(1)
    
    # Keep trying to SSH until we're on the target node
    max_attempts = 10
    attempt = 0
    
    while attempt < max_attempts:
        # Get the last line to check current hostname
        captured_lines = pane.capture_pane()
        if captured_lines:
            last_line = captured_lines[-1].strip()
            if last_line == "$":
                last_line = captured_lines[-2].strip() if len(captured_lines) > 1 else ""
        else:
            last_line = ""

        # Check if we're already on the target node
        if node in last_line:
            print(f"Successfully connected to {node}")
            break
        else:
            print(f"Attempt {attempt + 1}: SSHing to {node}")
            pane.send_keys(f'ssh {node}', enter=True)
            time.sleep(3)  # Wait for SSH connection to establish
            attempt += 1
    
    if attempt >= max_attempts:
        print(f"Failed to connect to {node} after {max_attempts} attempts")

Attempt 1: SSHing to research-secure-04
Successfully connected to research-secure-04
Waiting for pane to be ready: 06
Successfully connected to research-secure-06


In [13]:


CMD = """
sudo usermod -aG docker $USER
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")



  ✅ Command sent to 04
  ✅ Command sent to 06


In [14]:
CMD = """
newgrp docker
"""


# Send vllm serve command to each window
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(CMD, enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 04
  ✅ Command sent to 06


In [15]:
# Send vllm serve command to each window
# Only do when the disk is full
for window in session.windows:
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys('sudo docker system prune -a --volumes --force', enter=True)
    print(f"  ✅ Command sent to {window_name}")

  ✅ Command sent to 04
  ✅ Command sent to 06


In [16]:
import datasets
dataset = datasets.load_dataset("togethercomputer/R2E-Gym-Openhands-SWE-Bench-Verified", split="test")
# dataset = datasets.load_dataset("togethercomputer/R2E-Gym-Openhands-SWE-Bench-Pro", split="train")

/data/jisenli2/miniconda/envs/r2egym/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [17]:
dataset

Dataset({
    features: ['repo', 'instance_id', 'base_commit', 'patch', 'test_patch', 'problem_statement', 'hints_text', 'created_at', 'version', 'FAIL_TO_PASS', 'PASS_TO_PASS', 'environment_setup_commit', 'parsed_commit', 'run_tests', 'docker_image'],
    num_rows: 500
})

In [18]:
CMD_TEMPLATE = """
source /data/jisenli2/miniconda/etc/profile.d/conda.sh
conda activate r2egym
cd /data/jisenli2/R2E-Gym

python src/r2egym/run/eval_openhands.py \
  --max_workers 24 \
  --dataset_num_shards {num_nodes} \\
  --dataset_shard_index {node_idx} \\
  --dataset "togethercomputer/R2E-Gym-Openhands-SWE-Bench-Verified" \
  --split "test" \
  --llm_name "openai/Qwen3_4B" \
  --base_url "http://localhost:{base_port}/v1" \
  --api_key "sk-dummy-key-for-local-model" \
  --use_fn_calling True \
  --exp_name {exp_name} \
  --output_dir /data/jisenli2/kv_rotation/swe_bench_verified_results/{exp_name} \
  --temperature 0.7 \
  --top_p 0.8 \
  --max_steps 100 \
  --backend "docker" \
  --max_reward_calc_time 1200 \
  --env_mode "swebench" \
  --max_output_tokens 32678 \
  --pass_n {pass_n}
"""

pass_n = 1

# Calculate dataset sharding for 3 nodes
total_examples = len(dataset)
num_nodes = len(nodes)

# Create commands for each node
commands = []
for node_idx in range(num_nodes):
    cmd = CMD_TEMPLATE.format(node_idx=node_idx, num_nodes=num_nodes, pass_n=pass_n, exp_name=exp_name, base_port=base_port)
    commands.append(cmd)
    print(f"Node {node_idx}: num_nodes={num_nodes}")


# Send vllm serve command to each window
for idx, window in enumerate(session.windows):
    window_name = window.name
    pane = window.active_pane
    
    pane.send_keys(commands[idx], enter=True)
    print(f"  ✅ Command sent to {window_name}")

Node 0: num_nodes=2
Node 1: num_nodes=2
  ✅ Command sent to 04
  ✅ Command sent to 06


# results collects

In [6]:
import os
import tqdm
import json
import glob
import tqdm
import datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from typing import List, Dict, Optional
def print_reward_files(path):
    reward_files = glob.glob(os.path.join(path, "**", "reward.json"))

    from concurrent.futures import ThreadPoolExecutor, as_completed
    import json

    def read_reward_file(file):
        with open(file, "r") as f:
            return json.load(f)

    with ThreadPoolExecutor(max_workers=64) as executor:
        futures = {executor.submit(read_reward_file, file): file for file in reward_files}
        all_data = []
        for future in tqdm.tqdm(as_completed(futures), total=len(reward_files), desc="Reading reward files"):
            all_data.append(future.result())

    print(len(all_data))
    rewards = [data["reward"] for data in all_data]
    print(np.mean(rewards))
    # 1 / (9000 * 4 / len(reward_files))
    # 4500 * 8

In [ ]:
path = f"/data/jisenli2/kv_rotation/swe_bench_verified_results/{exp}/togethercomputer_R2E-Gym-Openhands-SWE-Bench-Verified/OpenhandsAgent/openai_Qwen3_4B_max100_temp0.7_topp0.8/run1"
print_reward_files(path)

Reading reward files: 100%|██████████| 25/25 [00:00<00:00, 5562.74it/s]

25
0.0
